In [ ]:
import os
from google.colab import drive
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

In [ ]:

import os
from google.colab import drive
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# --- STEP 1: MOUNT GOOGLE DRIVE ---
# This will open a popup asking for permission to access your files.
drive.mount('/content/drive')

# --- STEP 2: CONFIGURATION ---
# Replace with your actual Groq API Key
os.environ["GROQ_API_KEY"] = "you groq api"

# Path to your existing vector database on Google Drive
# Standard path usually starts with '/content/drive/MyDrive/'
db_path = "/content/drive/MyDrive/psych_vector_db"

# --- STEP 3: INITIALIZE COMPONENTS ---

# 1. Embedding Model (Needed to translate your questions into math)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 2. Load Vector DB from Drive
if os.path.exists(db_path):
    vector_db = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )
    print("✅ Database successfully loaded from Google Drive.")
else:
    print(f"❌ Error: Could not find '{db_path}'. Please check your folder name in Drive.")

# 3. Initialize LLM (Llama 3.1 8B for maximum intelligence)
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0
)

# --- STEP 4: DEFINE THE BRAIN (The Prompt) ---
template = """
You are a professional Psychology Research Assistant.
Use the Context below to answer the user's Question.

If the Question is just a greeting (like 'hello'), greet them back and ask how you can help.
If the answer is not in the Context, tell the user the information isn't in your current database.

Context:
{context}

Question: {question}

Helpful Answer:"""

custom_prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# --- STEP 5: CREATE THE CHAIN (The Loop-Proof Engine) ---
# RetrievalQA prevents the "Iteration Limit" error by avoiding complex agent loops.
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": custom_prompt},
    return_source_documents=True
)

# --- STEP 6: EXECUTION ---
def run_psych_assistant():
    print("\n--- LAU Psychology IPMS is Active ---")
    while True:
        query = input("\nYou: ")
        if query.lower() in ["exit", "quit", "stop"]:
            print("Shutting down...")
            break

        try:
            response = qa_chain.invoke({"query": query})
            print(f"\nAssistant: {response['result']}")

        except Exception as e:
            if "rate_limit" in str(e).lower():
                print("⚠️ Rate limit reached on Groq. Please wait 10 seconds.")
            else:
                print(f"⚠️ An error occurred: {e}")

if __name__ == "__main__":
    run_psych_assistant()